# Notebook 05 — Final Load Preparation for Tableau

**Project:** Why Startups Fail — VC Investment Pattern Analysis  
**Team:** Section C, Group 17  

---
**Objective:** Produce the final, Tableau-optimised output files from the cleaned dataset. This includes KPI summary tables, aggregated views, and the master export with all engineered features. All outputs are committed to `data/processed/`.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

PROCESSED_DIR = '/content/startups_cleaned.csv'

df = pd.read_csv('/content/startups_cleaned.csv', low_memory=False)
print(f'Loaded cleaned data: {df.shape[0]:,} rows x {df.shape[1]} columns')

Loaded cleaned data: 28,534 rows x 47 columns


In [2]:
# Select and rename columns for Tableau clarity
tableau_cols = [
    'name', 'status', 'country_code', 'market', 'primary_category',
    'founding_decade', 'founded_year', 'founded_at', 'first_funding_at', 'last_funding_at',
    'funding_rounds', 'funding_total_usd', 'avg_funding_per_round',
    'seed', 'venture', 'angel', 'round_A', 'round_B', 'round_C',
    'days_to_first_funding', 'funding_duration_days',
    'is_closed', 'is_usa', 'reached_series_a'
]

df_tableau = df[[c for c in tableau_cols if c in df.columns]].copy()

# Rename for Tableau display
rename_map = {
    'name': 'Startup Name',
    'status': 'Status',
    'country_code': 'Country',
    'market': 'Market',
    'primary_category': 'Primary Category',
    'founding_decade': 'Founding Decade',
    'founded_year': 'Founded Year',
    'founded_at': 'Founded Date',
    'first_funding_at': 'First Funding Date',
    'last_funding_at': 'Last Funding Date',
    'funding_rounds': 'Funding Rounds',
    'funding_total_usd': 'Total Funding USD',
    'avg_funding_per_round': 'Avg Funding Per Round USD',
    'seed': 'Seed Amount USD',
    'venture': 'Venture Amount USD',
    'angel': 'Angel Amount USD',
    'round_A': 'Series A USD',
    'round_B': 'Series B USD',
    'round_C': 'Series C USD',
    'days_to_first_funding': 'Days to First Funding',
    'funding_duration_days': 'Funding Duration Days',
    'is_closed': 'Closed (0/1)',
    'is_usa': 'USA Based (0/1)',
    'reached_series_a': 'Reached Series A (0/1)'
}

df_tableau.rename(columns=rename_map, inplace=True)

out_path = f'{PROCESSED_DIR}tableau_master.csv'
df_tableau.to_csv(out_path, index=False)
print(f'Master Tableau dataset exported: {out_path}')
print(f'Shape: {df_tableau.shape[0]:,} rows x {df_tableau.shape[1]} columns')
df_tableau.head(3)

Master Tableau dataset exported: /content/startups_cleaned.csvtableau_master.csv
Shape: 28,534 rows x 24 columns


,Startup Name,Status,Country,Market,Primary Category,Founding Decade,Founded Year,Founded Date,First Funding Date,Last Funding Date,...,Venture Amount USD,Angel Amount USD,Series A USD,Series B USD,Series C USD,Days to First Funding,Funding Duration Days,Closed (0/1),USA Based (0/1),Reached Series A (0/1)
0,#waywire,acquired,USA,News,Entertainment,2010,2012.0,2012-06-01,2012-06-30,2012-06-30,...,0,0,0,0,0,29.0,0.0,0,1,0
1,(In)Touch Network,operating,GBR,Electronics,Electronics,2010,2011.0,2011-04-01,2011-04-01,2011-04-01,...,0,0,0,0,0,0.0,0.0,0,0,0
2,-R- Ranch and Mine,operating,USA,Tourism,Tourism,2010,2014.0,2014-01-01,2014-08-17,2014-09-26,...,0,0,0,0,0,228.0,40.0,0,1,0


In [3]:
total = len(df)

kpi_data = {
    'KPI': [
        'Total Startups Analysed',
        'Overall Failure Rate (%)',
        'Operating Rate (%)',
        'Acquisition Rate (%)',
        'IPO Rate (%)',
        'Median Funding — Closed ($)',
        'Median Funding — Operating ($)',
        'Funding Gap (Operating vs Closed) ($)',
        'Avg Funding Rounds — Closed',
        'Avg Funding Rounds — Operating',
        'Avg Funding Rounds — Acquired',
        'Series A Post-Funding Failure Rate (%)',
        'Pre-Series A Failure Rate (%)',
        'USA Startup Failure Rate (%)',
        'Non-USA Startup Failure Rate (%)'
    ],
    'Value': [
        total,
        round((df['status'] == 'closed').sum() / total * 100, 2),
        round((df['status'] == 'operating').sum() / total * 100, 2),
        round((df['status'] == 'acquired').sum() / total * 100, 2),
        round((df['status'] == 'ipo').sum() / total * 100, 2),
        int(df[df['status'] == 'closed']['funding_total_usd'].median()),
        int(df[df['status'] == 'operating']['funding_total_usd'].median()),
        int(df[df['status'] == 'operating']['funding_total_usd'].median() -
            df[df['status'] == 'closed']['funding_total_usd'].median()),
        round(df[df['status'] == 'closed']['funding_rounds'].mean(), 2),
        round(df[df['status'] == 'operating']['funding_rounds'].mean(), 2),
        round(df[df['status'] == 'acquired']['funding_rounds'].mean(), 2),
        round(df[df['reached_series_a'] == 1]['is_closed'].mean() * 100, 2),
        round(df[df['reached_series_a'] == 0]['is_closed'].mean() * 100, 2),
        round(df[df['country_code'] == 'USA']['is_closed'].mean() * 100, 2),
        round(df[df['country_code'] != 'USA']['is_closed'].mean() * 100, 2)
    ]
}

df_kpi = pd.DataFrame(kpi_data)
kpi_path = f'{PROCESSED_DIR}kpi_summary.csv'
df_kpi.to_csv(kpi_path, index=False)
print(f'KPI summary exported: {kpi_path}')
print(df_kpi.to_string(index=False))

KPI summary exported: /content/startups_cleaned.csvkpi_summary.csv
                                   KPI      Value
               Total Startups Analysed   28534.00
              Overall Failure Rate (%)       5.08
                    Operating Rate (%)      86.54
                  Acquisition Rate (%)       8.38
                          IPO Rate (%)       0.00
           Median Funding — Closed ($)  863880.00
        Median Funding — Operating ($) 1800000.00
 Funding Gap (Operating vs Closed) ($)  936120.00
           Avg Funding Rounds — Closed       1.53
        Avg Funding Rounds — Operating       1.92
         Avg Funding Rounds — Acquired       2.23
Series A Post-Funding Failure Rate (%)       4.59
         Pre-Series A Failure Rate (%)       5.33
          USA Startup Failure Rate (%)       4.97
      Non-USA Startup Failure Rate (%)       5.28


In [4]:
country_agg = df.groupby('country_code').agg(
    Total_Startups=('is_closed', 'count'),
    Closed=('is_closed', 'sum'),
    Avg_Funding_USD=('funding_total_usd', 'mean'),
    Median_Funding_USD=('funding_total_usd', 'median'),
    Avg_Rounds=('funding_rounds', 'mean')
).reset_index()
country_agg['Failure_Rate_Pct'] = (country_agg['Closed'] / country_agg['Total_Startups'] * 100).round(2)
country_agg = country_agg[country_agg['Total_Startups'] >= 30].sort_values('Failure_Rate_Pct', ascending=False)

country_path = f'{PROCESSED_DIR}country_level_summary.csv'
country_agg.to_csv(country_path, index=False)
print(f'Country summary exported: {country_path}')
print(country_agg.head(15).to_string(index=False))

Country summary exported: /content/startups_cleaned.csvcountry_level_summary.csv
country_code  Total_Startups  Closed  Avg_Funding_USD  Median_Funding_USD  Avg_Rounds  Failure_Rate_Pct
         EST              32       4     2.191053e+06            553288.0    1.718750             12.50
         NZL              38       4     5.155439e+06           1050000.0    1.710526             10.53
         RUS             136      14     6.926827e+06           1269000.0    1.713235             10.29
         AUT              56       5     3.932943e+06           1193300.0    1.821429              8.93
         MYS              35       3     7.347297e+06            500000.0    1.685714              8.57
         NOR              55       4     4.919227e+06           1600000.0    1.490909              7.27
         BEL              89       6     5.741142e+06           1250000.0    1.651685              6.74
         ISR             489      31     8.316885e+06           3750000.0    1.697342  

In [5]:
sector_agg = df.groupby('market').agg(
    Total_Startups=('is_closed', 'count'),
    Closed=('is_closed', 'sum'),
    Avg_Funding_USD=('funding_total_usd', 'mean'),
    Avg_Rounds=('funding_rounds', 'mean'),
    Series_A_Rate=('reached_series_a', 'mean')
).reset_index()
sector_agg['Failure_Rate_Pct'] = (sector_agg['Closed'] / sector_agg['Total_Startups'] * 100).round(2)
sector_agg['Series_A_Rate_Pct'] = (sector_agg['Series_A_Rate'] * 100).round(2)
sector_agg = sector_agg[sector_agg['Total_Startups'] >= 50].sort_values('Failure_Rate_Pct', ascending=False)

sector_path = f'{PROCESSED_DIR}sector_level_summary.csv'
sector_agg.to_csv(sector_path, index=False)
print(f'Sector summary exported: {sector_path}')
print(sector_agg.head(20).to_string(index=False))

Sector summary exported: /content/startups_cleaned.csvsector_level_summary.csv
              market  Total_Startups  Closed  Avg_Funding_USD  Avg_Rounds  Series_A_Rate  Failure_Rate_Pct  Series_A_Rate_Pct
    Public Relations              87      17     6.977096e+06    1.436782       0.264368             19.54              26.44
     Web Development              79      11     4.568354e+06    1.455696       0.303797             13.92              30.38
         Curated Web             960     132     4.502552e+06    1.655208       0.268750             13.75              26.88
Social Network Media             113      14     3.833572e+06    1.814159       0.247788             12.39              24.78
             Android              59       7     3.532124e+06    1.949153       0.288136             11.86              28.81
          Networking             107      12     7.420243e+06    1.747664       0.345794             11.21              34.58
        Social Media             528   

In [6]:
yearly_agg = df[df['founded_year'].between(1990, 2022)].groupby('founded_year').agg(
    Total_Startups=('is_closed', 'count'),
    Closed=('is_closed', 'sum'),
    Avg_Funding_USD=('funding_total_usd', 'mean'),
    Avg_Rounds=('funding_rounds', 'mean')
).reset_index()
yearly_agg['Failure_Rate_Pct'] = (yearly_agg['Closed'] / yearly_agg['Total_Startups'] * 100).round(2)

yearly_path = f'{PROCESSED_DIR}yearly_trend_summary.csv'
yearly_agg.to_csv(yearly_path, index=False)
print(f'Yearly trend exported: {yearly_path}')
print(yearly_agg.to_string(index=False))

Yearly trend exported: /content/startups_cleaned.csvyearly_trend_summary.csv
 founded_year  Total_Startups  Closed  Avg_Funding_USD  Avg_Rounds  Failure_Rate_Pct
       1990.0              70       3     1.327636e+07    1.457143              4.29
       1991.0              75       1     1.518452e+07    1.586667              1.33
       1992.0              92       2     1.369455e+07    1.576087              2.17
       1993.0             105       4     1.377889e+07    1.723810              3.81
       1994.0             129       4     1.538299e+07    1.782946              3.10
       1995.0             169       2     1.550592e+07    1.857988              1.18
       1996.0             249       8     1.536420e+07    1.939759              3.21
       1997.0             310       9     1.521830e+07    1.880645              2.90
       1998.0             356       9     1.553469e+07    2.078652              2.53
       1999.0             655      32     1.605512e+07    2.155725       

In [7]:
import os

outputs = [
    ('tableau_master.csv', 'Main dashboard feed — all startup records'),
    ('kpi_summary.csv', 'Executive KPI table for dashboard header'),
    ('country_level_summary.csv', 'Geographic map view'),
    ('sector_level_summary.csv', 'Sector drill-down view'),
    ('yearly_trend_summary.csv', 'Time trend line chart'),
]

print('=== FINAL LOAD SUMMARY ===')
for fname, desc in outputs:
    path = f'{PROCESSED_DIR}{fname}'
    if os.path.exists(path):
        rows = sum(1 for _ in open(path)) - 1
        size_kb = os.path.getsize(path) / 1024
        print(f'  ✓ {fname:<40} {rows:>6,} rows | {size_kb:>6.1f} KB | {desc}')
    else:
        print(f'  ✗ {fname} — NOT FOUND')

print()
print('All files are ready for Tableau import.')
print('Connect tableau_master.csv as the primary data source.')
print('Use the summary files for pre-aggregated calculated fields and KPI tiles.')

=== FINAL LOAD SUMMARY ===
  ✓ tableau_master.csv                       28,534 rows | 4383.7 KB | Main dashboard feed — all startup records
  ✓ kpi_summary.csv                              15 rows |    0.5 KB | Executive KPI table for dashboard header
  ✓ country_level_summary.csv                    37 rows |    2.2 KB | Geographic map view
  ✓ sector_level_summary.csv                     67 rows |    5.6 KB | Sector drill-down view
  ✓ yearly_trend_summary.csv                     25 rows |    1.5 KB | Time trend line chart

All files are ready for Tableau import.
Connect tableau_master.csv as the primary data source.
Use the summary files for pre-aggregated calculated fields and KPI tiles.
